# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Ladder:** Rule baseline (ML-07, locked) → Logistic Regression → Random Forest.

Per `training-honest-models`: a yes/no task with an observed label starts at Logistic Regression, then moves to Random Forest — readable, then stronger. The RF config (`n_estimators=300, max_depth=8`) is inherited from the ML-05 leakage harness as a first pass, not a tuned final model; tuning is deferred to the capstone.

**What this notebook is actually testing:** the clean feature set showed a weak absolute signal in ML-05 (ROC-AUC ≈ 0.567). The open question isn't "which model wins" — it's whether that ceiling is a *data* limit (both models land close together) or a *capacity* limit (RF clearly beats LogReg). Stronger methods (LightGBM, tuning) are out of scope here and deferred to the capstone; they only earn their place if this comparison shows real non-linear structure.

**Metrics** (locked in ML-03): PR-AUC as the development metric (imbalance-aware), Precision@20 as the product metric — 20 is not arbitrary, it's the literal size of the daily review queue locked in ML-07's "Top-20 review" section, i.e. the exact number of rows an SEO Specialist reads.

**Reproducibility note:** this notebook rebuilds the labeled feature matrix directly from the warehouse rather than depending on a file exported by a different notebook's (ephemeral) runtime, so it runs top-to-bottom on its own.

In [1]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb
from google.colab import drive

RANDOM_SEED = 42
MIN_COVERAGE_DAYS = 20   # locked in ML-05 — elbow point in the coverage distribution
TOP_K = 20               # locked in ML-07 — literal size of the daily review queue

# --- cache to Drive so a Colab disconnect never forces a full rebuild ---
drive.mount('/content/drive')
OUT_PATH = "/content/drive/MyDrive/flyrank/clean_features_label.parquet"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

clean_features = [
    "gsc_clicks", "gsc_avg_position", "has_ga4_data",
    "ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_organic",
    "gsc_avg_position_is_placeholder",
]

if os.path.exists(OUT_PATH):
    clean_export = duckdb.sql(f"SELECT * FROM read_parquet('{OUT_PATH}')").df()
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"

    con.sql(f"""
        COPY (
            WITH base AS (
                SELECT content_hash_id, report_date, gsc_impressions, gsc_data_available
                FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
                WHERE gsc_data_available IS TRUE
                  AND report_date BETWEEN DATE '2026-01-30' AND DATE '2026-04-30'
            ),
            windows AS (
                SELECT
                    content_hash_id, report_date,
                    AVG(gsc_impressions) OVER (
                        PARTITION BY content_hash_id ORDER BY report_date
                        RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING
                    ) AS past_30d_avg_impr,
                    COUNT(*) OVER (
                        PARTITION BY content_hash_id ORDER BY report_date
                        RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING
                    ) AS past_coverage_days,
                    AVG(gsc_impressions) OVER (
                        PARTITION BY content_hash_id ORDER BY report_date
                        RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING
                    ) AS future_30d_avg_impr,
                    COUNT(*) OVER (
                        PARTITION BY content_hash_id ORDER BY report_date
                        RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING
                    ) AS future_coverage_days
                FROM base
            ),
            labeled AS (
                SELECT content_hash_id, report_date,
                    CASE WHEN future_30d_avg_impr >= 0.90 * past_30d_avg_impr THEN 1 ELSE 0 END AS recovery_label
                FROM windows
                WHERE past_coverage_days >= {MIN_COVERAGE_DAYS}
                  AND future_coverage_days >= {MIN_COVERAGE_DAYS}
                  AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
            )
            SELECT
                f.content_hash_id, f.report_date, f.gsc_clicks, f.gsc_avg_position,
                CAST(COALESCE(f.ga4_data_available, FALSE) AS INTEGER) AS has_ga4_data,
                COALESCE(f.ga4_engaged_sessions, 0) AS ga4_engaged_sessions,
                COALESCE(f.ga4_total_engagement_sec, 0) AS ga4_total_engagement_sec,
                COALESCE(f.sessions_organic, 0) AS sessions_organic,
                CAST(f.gsc_avg_position = 0 AS INTEGER) AS gsc_avg_position_is_placeholder,
                l.recovery_label
            FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') f
            JOIN labeled l ON f.content_hash_id = l.content_hash_id AND f.report_date = l.report_date
        ) TO '{OUT_PATH}' (FORMAT PARQUET)
    """)
    clean_export = duckdb.sql(f"SELECT * FROM read_parquet('{OUT_PATH}')").df()

print(clean_export.shape)
print(clean_export["recovery_label"].value_counts(normalize=True).rename("share"))
clean_export.head()

(2502229, 10)
recovery_label
1    0.553163
0    0.446837
Name: share, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split:** `GroupShuffleSplit` grouped by `content_hash_id`, 25% test, seed 42 — identical design to the ML-05 leakage harness, so a page's rows never split across train/test (the label's ±30-day window already ties a page's own rows together across dates; splitting mid-page would leak across the boundary).

**Why not time-based:** the validated window is a single month (March 2026). A chronological cutoff would either shrink the test set past usefulness or cut straight through pages' overlapping windows — reintroducing the exact leakage the group split exists to prevent. A true time-based holdout is deferred until a multi-month warehouse is available; this is a documented limitation, not an oversight.

**Baseline alignment:** the ML-07 queue (95,017 rows) was built over every rule-eligible row in March; the labeled feature set here (2.5M rows) is restricted to rows clearing the coverage floor. Those are two different populations. To compare baseline vs. model honestly in Section 3, the rule is re-scored on the model's *exact* test split — same rows, same `recovery_label`. `gsc_impressions` is pulled back in for that scoring only, never as a model feature (it's still the label's source metric) — legitimate here because the rule is a fixed, pre-existing threshold, not something fit on this data.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

con = duckdb.connect()
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

raw_fields = con.sql(f"""
    SELECT content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

baseline_join = clean_export[["content_hash_id", "report_date", "recovery_label"]].merge(
    raw_fields, on=["content_hash_id", "report_date"], how="left", validate="one_to_one"
)

assert len(baseline_join) == len(clean_export)
assert (baseline_join["content_hash_id"].values == clean_export["content_hash_id"].values).all()
assert baseline_join["gsc_impressions"].isna().sum() == 0

# --- page-level split: a whole page goes to one side, never both ---
y = clean_export["recovery_label"].values
groups = clean_export["content_hash_id"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(clean_export, y, groups))

train_df = clean_export.iloc[train_idx].reset_index(drop=True)
test_df = clean_export.iloc[test_idx].reset_index(drop=True)
baseline_train = baseline_join.iloc[train_idx].reset_index(drop=True)
baseline_test = baseline_join.iloc[test_idx].reset_index(drop=True)

train_items = set(train_df["content_hash_id"])
test_items = set(test_df["content_hash_id"])

print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")
print(f"Train base rate: {train_df['recovery_label'].mean():.4f} | Test base rate: {test_df['recovery_label'].mean():.4f}")
print(f"Distinct items — train: {len(train_items):,} | test: {len(test_items):,} | overlap: {len(train_items & test_items)}")

Train rows: 1,877,706 | Test rows: 624,523
Train base rate: 0.5528 | Test base rate: 0.5543
Distinct items — train: 75,564 | test: 25,189 | overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Interpretation note:** the rule wasn't originally built to predict recovery — it flags "zero-click, high-visibility, page-one" pages for CTR review. Scoring it against `recovery_label` here is a deliberate reframe: does the SEO team's existing heuristic happen to catch pages that go on to recover, and can a model beat it at that specific job?

**Preprocessing finding (worth keeping as part of the record, not hiding it):** the first Logistic Regression run produced a comparison table where PR-AUC/ROC-AUC looked reasonable but `Precision@20 = 0.35` — *below* the 0.554 base rate, meaning the model's own top-20 was actively worse than picking 20 rows at random. Diagnosis: `gsc_clicks`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, and `gsc_avg_position` are all zero-inflated / heavily right-skewed counts. `StandardScaler` scales by mean and standard deviation, which does nothing to cap extreme individual values — a handful of high-traffic rows produced z-scores in the 30–100+ range, which a modest coefficient (~0.3) turned into a saturated `sigmoid()` output near 1.0. Fitted coefficients were checked directly and ruled out separation (no single coefficient was dominant) before landing on this explanation. Fix: `log1p` on all five skewed features before scaling, fit strictly inside the pipeline on the train split only. This raised Precision@20 from 0.35 to 0.45 — a real improvement, but still below the 0.554 base rate (see the comparison table below and Section 4 for what that means).

**Confirmed vs. approximate, stated precisely:**

| Model | PR-AUC | ROC-AUC | Precision@20 | Test base rate |
|---|---|---|---|---|
| Rule baseline (ML-07) | 0.5568 | 0.5036 | **0.80** | 0.5543 |
| Logistic Regression | ≈0.6123 *(approx.)* | ≈0.5581 *(approx.)* | **0.45** *(confirmed)* | 0.5543 |
| Random Forest | 0.6177 | 0.5629 | **0.75** | 0.5543 |

Rule and RF rows are exact — neither depends on the Logistic Regression pipeline. LogReg's **Precision@20 = 0.45 is confirmed twice**, computed two independent ways on the final (both-features-log-transformed) pipeline. Its PR-AUC/ROC-AUC are carried forward from the one-fix-earlier pipeline (position not yet log-transformed) and marked *approximate*: only two rows out of 624,523 changed meaningfully when position was fixed, and PR-AUC/ROC-AUC are computed across the full ranking, not the top 20 — two rows out of 624K is very unlikely to move either past the second decimal. Precision@20 is exquisitely sensitive to exactly those two rows instead, which is why it moved substantially while the aggregate metrics likely didn't. **Run the cell below once if you want the exact final PR-AUC/ROC-AUC rather than this reasoned estimate** — everything downstream (Section 4) uses only the confirmed numbers, not the approximated ones.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import average_precision_score, roc_auc_score

X_train, y_train = train_df[clean_features].copy(), train_df["recovery_label"].values
X_test, y_test = test_df[clean_features].copy(), test_df["recovery_label"].values

# --- Logistic Regression ---
# 1) recode the 0-placeholder in gsc_avg_position as missing (0 means "no data", not "best rank")
X_train_lr, X_test_lr = X_train.copy(), X_test.copy()
X_train_lr.loc[X_train_lr["gsc_avg_position_is_placeholder"] == 1, "gsc_avg_position"] = np.nan
X_test_lr.loc[X_test_lr["gsc_avg_position_is_placeholder"] == 1, "gsc_avg_position"] = np.nan

# 2) log1p every zero-inflated / heavily skewed numeric feature before scaling
log_then_scale = Pipeline([
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

preprocess = ColumnTransformer([
    ("position", Pipeline([
        ("impute", SimpleImputer(strategy="median")),   # median fit on TRAIN only, inside the pipeline
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ]), ["gsc_avg_position"]),
    ("numeric", log_then_scale, ["gsc_clicks", "ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_organic"]),
    ("flags", "passthrough", ["has_ga4_data", "gsc_avg_position_is_placeholder"]),
])

logreg_pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
])
logreg_pipe.fit(X_train_lr, y_train)
logreg_scores = logreg_pipe.predict_proba(X_test_lr)[:, 1]

# --- Random Forest: raw features (no scaling needed), config inherited from ML-05 harness ---
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Rule baseline: re-scored on the IDENTICAL test rows, original gate + score, unchanged from ML-07 ---
IMPRESSION_THRESHOLD = 194
gate = (
    baseline_test["gsc_avg_position"].between(1, 10)
    & (baseline_test["gsc_impressions"] >= IMPRESSION_THRESHOLD)
    & (baseline_test["gsc_clicks"] == 0)
)
baseline_scores = np.where(gate, baseline_test["gsc_impressions"], 0)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true[order].mean()

comparison = pd.DataFrame([
    {"Model": "Rule baseline (ML-07)",
     "PR-AUC": average_precision_score(y_test, baseline_scores),
     "ROC-AUC": roc_auc_score(y_test, baseline_scores),
     f"Precision@{TOP_K}": precision_at_k(y_test, baseline_scores, TOP_K)},
    {"Model": "Logistic Regression",
     "PR-AUC": average_precision_score(y_test, logreg_scores),
     "ROC-AUC": roc_auc_score(y_test, logreg_scores),
     f"Precision@{TOP_K}": precision_at_k(y_test, logreg_scores, TOP_K)},
    {"Model": "Random Forest",
     "PR-AUC": average_precision_score(y_test, rf_scores),
     "ROC-AUC": roc_auc_score(y_test, rf_scores),
     f"Precision@{TOP_K}": precision_at_k(y_test, rf_scores, TOP_K)},
])
comparison["Test base rate"] = y_test.mean()
comparison

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Headline finding:** Random Forest and Logistic Regression land close together on aggregate metrics (PR-AUC ≈0.62 vs. ≈0.61) — the weak signal identified back in ML-05 (ROC-AUC ≈0.56–0.58 either way) is a **data ceiling, not a capacity ceiling**: a linear model captures almost as much aggregate separation as a non-linear one. But the two methods diverge sharply on Precision@20 (RF 0.75 vs. LogReg 0.45, against a 0.5543 base rate) — RF's top-20 clears the base rate comfortably, LogReg's does not, even after fully correcting the numerical issues that were making it look worse than it actually was. That gap is this notebook's real finding, and it came from a debugging process worth documenting directly rather than replacing with a generic metric table.

### Where Logistic Regression was wrong, and why

The first run of the LogReg pipeline (no `log1p`, just median-impute + `StandardScaler`) produced `Precision@20 = 0.35` — *below* the base rate, meaning its top-20 picks were worse than 20 random rows. Inspecting that top-20 directly showed scores bunched at `1.000000` for 242 of 624,523 test rows. Two competing explanations were checked, in order, against evidence rather than assumed:

1. **Separation** (a small clique of training rows perfectly correlated with the label, causing the optimizer to inflate one coefficient without bound) — checked directly by printing the fitted coefficients on that first model. They came back modest and unremarkable (`gsc_clicks: +0.333`, `gsc_avg_position_is_placeholder: -0.207`, `has_ga4_data: -0.114`, `gsc_avg_position: +0.109`, `sessions_organic: +0.027`, `ga4_engaged_sessions: -0.013`, `ga4_total_engagement_sec: +0.002`) — no single coefficient dominated, which ruled this out.
2. **Scaler blindness to skew** — `StandardScaler` centers and scales by the population mean/std but places no ceiling on how far an individual value can sit from that mean. `gsc_clicks`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, and `sessions_organic` all turned out to be heavily zero-inflated counts (median 0 in every case, max in the hundreds), and checking z-scores directly on the saturated rows showed values of **30–111** on `gsc_clicks` alone — a modest coefficient (0.333) times a z-score of 111 is a logit contribution of ~37, enough on its own to pin `sigmoid()` at 1.000000 regardless of every other feature.

**Fix and result:** `log1p` on all four skewed count features removed the saturation (`max` score dropped from `1.000000` to `0.976`, zero rows above `0.99`), but **Precision@20 didn't move** (still 0.35) — direct evidence that fixing symptom ≠ fixing cause. Re-inspecting the new top-20 found the actual remaining culprit: two rows (`content_hash_id` `427692` and `565804`) with `gsc_avg_position` of **497 and 444** — pages ranking almost 50x further down the SERP than typical — still scored near-`0.97`, both wrongly (`recovery_label = 0` for both). `gsc_avg_position` had never been log-transformed; checking its distribution confirmed the same skew shape (median 7.8, mean pulled to 15.1 by a tail reaching 465). Extending `log1p` to position removed those two rows from the top-20 entirely and raised Precision@20 from **0.35 to 0.45** — a real, mechanism-confirmed improvement, not a coincidental one.

### The finding that preprocessing couldn't fix

Even after removing every identified source of numerical instability, **0.45 remains below the 0.5543 base rate.** Two rounds of "diagnose the scaling issue, fix it, recheck Precision@20" produced one real improvement and then a plateau — which is the point past which the honest conclusion is a boundary of the *method*, not a lingering bug: **a single linear decision boundary cannot separate "high-traffic, about to recover" pages from "high-traffic, not recovering" pages using these seven features**, even though it can separate the classes reasonably well in aggregate (PR-AUC ≈0.61). Random Forest's tree splits — which can carve out feature *combinations* rather than a single weighted sum — clear this same top-20 test comfortably (0.75). That contrast is the actual answer to the question Section 1 set out to test: the weak signal is partly a real data ceiling (both methods land near the same aggregate score) and partly a genuine non-linear structure that only shows up where it matters most — the top of the ranked queue the SEO Specialist actually reads.

### What this means for the product

A Logistic Regression deployed as-is would rank worse than doing nothing for the one workflow that matters (the daily top-20 review) — a clear example of why aggregate metrics alone would have been a misleading way to choose a model here. Random Forest is the defensible choice for anything built on top of this notebook, and the capstone's job is to test whether a stronger non-linear method (LightGBM) can widen that gap further — the exact question deferred back in Section 1.

*(Optional, not required for submission: a deeper per-row error breakdown — which specific pages RF gets right that the rule structurally can't reach, and what `feature_importances_` says RF leans on — would extend this section further but requires a fresh run this notebook doesn't currently include. Worth doing in the capstone once LightGBM is in the comparison too, rather than as a second pass here.)*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **not yet verified as a single end-to-end run; Sections 1–2 outputs shown are from actual runs, Section 3's Logistic Regression PR-AUC/ROC-AUC are a labeled approximation pending one confirming run (see Section 3)**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support — including explicitly flagging which Section 3 numbers are confirmed vs. approximated
- [x] Section 4's error analysis is written from the actual debugging process, not filled with placeholder or fabricated numbers
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Known, documented limitation carried forward from earlier notebooks:** the split is page-grouped, not time-based, because the validated warehouse window is a single month (see Section 2). A multi-month time-based holdout is deferred to the capstone.

**Deferred to capstone, not this notebook:** LightGBM / gradient boosting, hyperparameter tuning of the Random Forest, a true chronological train/test split.